# Delivery Demand Forecasting — Feature Engineering Pipeline

**Goal:** Transform raw order-level delivery data into a rich AOI-hour dataset for demand forecasting.

## Architecture: Order-First, Aggregate Second

The critical insight: the old pipeline aggregated orders into hourly counts **first**, then tried
to engineer features from those counts alone. This destroys signal. Each order carries operational
information — delivery duration, trip distance, courier identity, spatial coordinates — that
reveals *why* demand behaves the way it does, not just *what* it was.

**Pipeline phases:**

| Phase | What | Why |
|-------|------|-----|
| 1. Order-level extraction | duration, distance, courier features per order | Captures operational dynamics before aggregation destroys them |
| 2. Rich AOI-hour aggregation | count + stats of duration, distance, couriers, spatial spread | Each AOI-hour gets a multidimensional profile, not just a count |
| 3. Complete hourly grid | Zero-fill missing hours with operational NaN -> 0 | Model learns "no orders" as a distinct state |
| 4. Chronological split | Assign train/val/test by time | Enables train-only statistics below |
| 5. Train-only statistics | Peak hours, AOI busyness from train only | Prevents leakage from val/test distributions |
| 6. Calendar & cyclical | Deterministic from timestamp | No leakage risk |
| 7. Lag/rolling/momentum | Per-AOI temporal features on demand AND operational stats | Historical patterns in operations, not just volume |
| 8. Spatial neighbors | Distance-weighted demand from nearby AOIs | Goes beyond flat region averages |
| 9. Cleanup & save | Drop warm-up, order columns, validate | Production-ready output |

**Leakage rules enforced:**
- `is_nonzero` is **removed** — it is a direct function of the target
- `aoi_busyness`, `peak_hours`, and all data-derived statistics use **train split only**
- All lag/rolling features use `shift()` to exclude the current hour
- The chronological split is assigned **before** any train-conditional feature computation

## 1. Configuration & Imports

All tunable parameters in one place.

In [ ]:
import pandas as pd
import numpy as np
import warnings
import gc

warnings.filterwarnings('ignore', category=FutureWarning)

SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

# ======================================================================
# CONFIGURATION — edit only this block
# ======================================================================
CONFIG = {
    # Paths
    "data_path":   "delivery_jl.csv",
    "output_path": "lade_hourly_features.csv",

    # Data
    "data_year": "2022",

    # Date filtering (None = disabled)
    "date_start": None,
    "date_end":   None,
    "max_aois":   None,

    # Hourly bucketing
    "time_bucket": "h",

    # Chronological split ratios
    "train_ratio": 0.60,
    "val_ratio":   0.20,

    # Lag/rolling config
    "lags":         [1, 2, 3, 6, 12, 24, 48, 168],
    "roll_windows": [6, 24, 72, 168],

    # Spatial neighbor config
    "n_neighbors":     5,
    "neighbor_max_km": 10.0,

    # Outlier thresholds
    "max_duration_hours": 24,
    "max_distance_km":    100,
}

print('Configuration loaded.')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## 2. Load Raw Order Data & Quality Checks

In [ ]:
# Mount Google Drive if in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

df_raw = pd.read_csv(CONFIG['data_path'])
print(f'Raw data: {df_raw.shape[0]:,} orders x {df_raw.shape[1]} columns')
print(f'Columns: {list(df_raw.columns)}')

# Quality checks
print(f'\nMissing values:')
print(df_raw.isna().sum()[df_raw.isna().sum() > 0])

print(f'\nUnique counts:')
print(f'  Cities:   {df_raw["city"].nunique()}')
print(f'  Regions:  {df_raw["region_id"].nunique()}')
print(f'  AOIs:     {df_raw["aoi_id"].nunique()}')
print(f'  Couriers: {df_raw["courier_id"].nunique()}')
print(f'  Orders:   {df_raw["order_id"].nunique()}')

dup_orders = df_raw['order_id'].duplicated().sum()
print(f'  Duplicate order_ids: {dup_orders}')
if dup_orders > 0:
    df_raw = df_raw.drop_duplicates(subset='order_id', keep='first')
    print(f'  Deduplicated -> {len(df_raw):,} orders')

## 3. Timestamp Parsing

Parse both `accept_time` and `delivery_time` into proper datetimes.
Both are needed to compute delivery duration — a key operational feature.

In [ ]:
YEAR = CONFIG['data_year']
df = df_raw.copy()

df['accept_dt'] = pd.to_datetime(
    YEAR + '-' + df['accept_time'].astype(str),
    format='%Y-%m-%d %H:%M:%S', errors='coerce'
)
df['delivery_dt'] = pd.to_datetime(
    YEAR + '-' + df['delivery_time'].astype(str),
    format='%Y-%m-%d %H:%M:%S', errors='coerce'
)

n_bad_accept   = df['accept_dt'].isna().sum()
n_bad_delivery = df['delivery_dt'].isna().sum()
print(f'Unparsable accept_time:   {n_bad_accept}')
print(f'Unparsable delivery_time: {n_bad_delivery}')

df = df.dropna(subset=['accept_dt', 'delivery_dt']).copy()

# Optional date-range filtering
if CONFIG['date_start']:
    df = df[df['accept_dt'] >= CONFIG['date_start']]
if CONFIG['date_end']:
    df = df[df['accept_dt'] < pd.to_datetime(CONFIG['date_end']) + pd.Timedelta(days=1)]

print(f'Date range: {df["accept_dt"].min()} -> {df["accept_dt"].max()}')
print(f'Orders after filtering: {len(df):,}')

df['bucket_hour'] = df['accept_dt'].dt.floor(CONFIG['time_bucket'])

## 4. Order-Level Feature Extraction (BEFORE Aggregation)

**This is the key architectural change.** The old pipeline jumped straight to counting orders
per AOI-hour, discarding everything about individual deliveries. Here we extract features from
each order while the information still exists.

### Features extracted per order:

| Feature | Formula | Signal |
|---------|---------|--------|
| `delivery_duration_min` | delivery_time - accept_time | Operational load: long deliveries signal congestion or distant orders |
| `trip_distance_km` | Haversine(accept_gps -> delivery_gps) | Physical workload: farther trips = more courier time committed |
| `order_displacement_km` | Haversine(order lng/lat -> delivery_gps) | How far actual delivery is from listed order location |

**Why these matter for forecasting:**
An AOI-hour with 5 orders averaging 15min each is very different from 5 orders averaging 90min.
The first is a well-functioning zone; the second is overloaded. These features are **orthogonal
to demand count** — they capture *how* deliveries happen, not *how many*.

In [ ]:
# ======================================================================
# PHASE 1: ORDER-LEVEL FEATURE EXTRACTION
# ======================================================================

def haversine_km(lon1, lat1, lon2, lat2):
    '''Vectorized haversine distance in kilometers.'''
    R = 6371.0
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

# Delivery duration (minutes)
df['delivery_duration_min'] = (
    (df['delivery_dt'] - df['accept_dt']).dt.total_seconds() / 60.0
).astype('float32')

# Trip distance: courier GPS at accept -> courier GPS at delivery
df['trip_distance_km'] = haversine_km(
    df['accept_gps_lng'].values, df['accept_gps_lat'].values,
    df['delivery_gps_lng'].values, df['delivery_gps_lat'].values
).astype('float32')

# Order displacement: order location -> delivery GPS
df['order_displacement_km'] = haversine_km(
    df['lng'].values, df['lat'].values,
    df['delivery_gps_lng'].values, df['delivery_gps_lat'].values
).astype('float32')

# Outlier capping
max_dur = CONFIG['max_duration_hours'] * 60
max_dist = CONFIG['max_distance_km']

n_neg_dur = (df['delivery_duration_min'] < 0).sum()
n_long_dur = (df['delivery_duration_min'] > max_dur).sum()
n_far = (df['trip_distance_km'] > max_dist).sum()

df['delivery_duration_min'] = df['delivery_duration_min'].clip(lower=0, upper=max_dur)
df['trip_distance_km']      = df['trip_distance_km'].clip(lower=0, upper=max_dist)
df['order_displacement_km'] = df['order_displacement_km'].clip(lower=0, upper=max_dist)

print(f'Order-level features extracted:')
print(f'  delivery_duration_min: median={df["delivery_duration_min"].median():.1f} min')
print(f'  trip_distance_km:      median={df["trip_distance_km"].median():.2f} km')
print(f'  order_displacement_km: median={df["order_displacement_km"].median():.2f} km')
print(f'  Outliers capped: {n_neg_dur} negative dur, {n_long_dur} extreme dur, {n_far} extreme dist')

## 5. Rich AOI-Hour Aggregation

**Old approach:** `groupby().count()` — one number per AOI-hour.

**New approach:** Each AOI-hour gets a **multidimensional operational profile:**
- Demand count (preserved)
- Delivery duration distribution (mean, std, min, max)
- Trip distance distribution (mean, std, min, max)
- Courier workforce metrics (unique count, orders-per-courier)
- Spatial dispersion (coordinate variance within the hour)

**Why this matters:** A model trained only on counts cannot distinguish between
10 orders / 5 couriers / avg 20min (healthy zone) vs 10 orders / 2 couriers / avg 90min (overloaded).
Operational stats let the model learn these patterns.

In [ ]:
# ======================================================================
# PHASE 2: RICH AOI-HOUR AGGREGATION
# ======================================================================
print('Phase 2: Aggregating orders into AOI-hour level...')

GROUP_KEYS = ['city', 'region_id', 'aoi_id', 'aoi_type', 'bucket_hour']

agg_dict = {
    'order_id': 'count',
    'delivery_duration_min': ['mean', 'std', 'min', 'max'],
    'trip_distance_km':      ['mean', 'std', 'min', 'max'],
    'order_displacement_km': ['mean', 'std'],
    'courier_id': 'nunique',
    'lng': ['std', 'mean'],
    'lat': ['std', 'mean'],
}

demand_agg = df.groupby(GROUP_KEYS, as_index=False).agg(agg_dict)

# Flatten multi-level column names
flat_cols = []
for col in demand_agg.columns:
    if isinstance(col, tuple):
        if col[0] == 'order_id' and col[1] == 'count':
            flat_cols.append('demand_count')
        elif col[0] == 'courier_id':
            flat_cols.append('n_unique_couriers')
        elif col[1] == '':
            flat_cols.append(col[0])
        else:
            flat_cols.append(f'{col[0]}_{col[1]}')
    else:
        flat_cols.append(col)
demand_agg.columns = flat_cols

# Derived operational features
demand_agg['orders_per_courier'] = (
    demand_agg['demand_count'] / demand_agg['n_unique_couriers'].clip(lower=1)
).astype('float32')

demand_agg['duration_range'] = (
    demand_agg['delivery_duration_min_max'] - demand_agg['delivery_duration_min_min']
).astype('float32')

demand_agg['distance_range'] = (
    demand_agg['trip_distance_km_max'] - demand_agg['trip_distance_km_min']
).astype('float32')

# Spatial dispersion: combined lng/lat std
demand_agg['spatial_dispersion'] = np.sqrt(
    demand_agg['lng_std'].fillna(0)**2 + demand_agg['lat_std'].fillna(0)**2
).astype('float32')

demand_agg = demand_agg.drop(columns=['lng_std', 'lat_std', 'lng_mean', 'lat_mean'], errors='ignore')

print(f'Aggregated shape: {demand_agg.shape}')
print(f'Columns: {list(demand_agg.columns)}')
print(f'\nDemand distribution:')
print(demand_agg['demand_count'].describe())

## 6. Complete Hourly Grid (Zero-Fill)

Hours with no orders are missing from the aggregation. We build a complete grid so the model
explicitly learns what zero demand looks like. Operational stats are filled with 0.

In [ ]:
# Optional AOI selection
if CONFIG['max_aois'] is not None:
    aoi_totals = (
        demand_agg.groupby(['city', 'region_id', 'aoi_id', 'aoi_type'])['demand_count']
        .sum().sort_values(ascending=False)
        .head(CONFIG['max_aois']).reset_index()
    )
    aoi_keys = aoi_totals[['city', 'region_id', 'aoi_id', 'aoi_type']]
else:
    aoi_keys = demand_agg[['city', 'region_id', 'aoi_id', 'aoi_type']].drop_duplicates().reset_index(drop=True)

print(f'AOIs selected: {len(aoi_keys)}')

# Build complete hourly grid
full_hours = pd.date_range(
    demand_agg['bucket_hour'].min(),
    demand_agg['bucket_hour'].max(),
    freq=CONFIG['time_bucket']
)
print(f'Full hour range: {full_hours.min()} -> {full_hours.max()} ({len(full_hours)} hours)')

# Cross join: every AOI x every hour
grid = aoi_keys.assign(_k=1).merge(
    pd.DataFrame({'bucket_hour': full_hours, '_k': 1}), on='_k'
).drop(columns='_k')

model_df = grid.merge(demand_agg, on=GROUP_KEYS, how='left')
model_df['demand_count'] = model_df['demand_count'].fillna(0).astype('int32')

# Fill operational cols with 0 for zero-demand hours
operational_cols = [c for c in model_df.columns
                    if c not in GROUP_KEYS + ['demand_count']
                    and model_df[c].dtype in ['float32', 'float64', 'int64']]
for c in operational_cols:
    model_df[c] = model_df[c].fillna(0).astype('float32')

model_df = model_df.sort_values(['city', 'region_id', 'aoi_id', 'bucket_hour']).reset_index(drop=True)

print(f'Grid shape: {model_df.shape}')
print(f'Zero-demand rows: {(model_df["demand_count"] == 0).sum():,} '
      f'({(model_df["demand_count"] == 0).mean():.1%})')

del grid
gc.collect()

## 7. Chronological Split Assignment (Early)

**Why assign splits now:** The next steps compute data-derived statistics (peak hours, AOI busyness)
that must use **train data only** to prevent leakage. By assigning the split label here, we can
filter to train rows for those computations.

In [ ]:
all_hours = np.sort(model_df['bucket_hour'].unique())
n_hours   = len(all_hours)

train_end = all_hours[int(n_hours * CONFIG['train_ratio']) - 1]
val_end   = all_hours[int(n_hours * (CONFIG['train_ratio'] + CONFIG['val_ratio'])) - 1]

model_df['split'] = np.where(
    model_df['bucket_hour'] <= train_end, 'train',
    np.where(model_df['bucket_hour'] <= val_end, 'val', 'test')
)

for s in ['train', 'val', 'test']:
    n = (model_df['split'] == s).sum()
    hours = model_df.loc[model_df['split'] == s, 'bucket_hour']
    print(f'{s:5s}: {n:>10,} rows | {hours.min()} -> {hours.max()}')

print(f'\nSplit boundaries: train_end={train_end}, val_end={val_end}')
TRAIN_MASK = model_df['split'] == 'train'
print(f'Train rows: {TRAIN_MASK.sum():,}')

## 8. Train-Only Derived Features (Leakage-Safe)

**These features are learned from data** — they summarize patterns across rows. If computed on
the full dataset, they encode information from val/test periods, creating subtle leakage.

**Computed on train split only, then applied to all rows:**
- **Peak hour detection** — which hours have above-average demand (learned from train)
- **AOI busyness** — each AOI baseline activity level (mean hourly demand in train)
- **AOI operational profile** — mean delivery duration and distance per AOI (train only)

**Removed:** `is_nonzero` — this is `1 if demand_count > 0 else 0`, a **direct function of the target**.
Including it would let the model trivially predict non-zero demand by reading this feature.

In [ ]:
print('Phase 5: Computing train-only derived features...')
train_data = model_df[TRAIN_MASK]

# Peak hours (train-only)
hourly_profile = train_data.groupby(train_data['bucket_hour'].dt.hour)['demand_count'].mean()
peak_threshold = hourly_profile.quantile(0.75)
peak_hours_set = set(hourly_profile[hourly_profile >= peak_threshold].index)
print(f'  Peak hours (from train): {sorted(peak_hours_set)}')
model_df['is_peak_hour'] = model_df['bucket_hour'].dt.hour.isin(peak_hours_set).astype('int8')

# AOI busyness (train-only)
aoi_busyness = (
    train_data.groupby('aoi_id')['demand_count']
    .mean().rename('aoi_busyness').reset_index()
)
model_df = model_df.merge(aoi_busyness, on='aoi_id', how='left')
model_df['aoi_busyness'] = model_df['aoi_busyness'].fillna(0).astype('float32')

# AOI operational profile (train-only)
aoi_ops = (
    train_data[train_data['demand_count'] > 0]
    .groupby('aoi_id')
    .agg(
        aoi_avg_duration=('delivery_duration_min_mean', 'mean'),
        aoi_avg_distance=('trip_distance_km_mean', 'mean'),
        aoi_avg_couriers=('n_unique_couriers', 'mean'),
    )
    .reset_index()
)
model_df = model_df.merge(aoi_ops, on='aoi_id', how='left')
for c in ['aoi_avg_duration', 'aoi_avg_distance', 'aoi_avg_couriers']:
    model_df[c] = model_df[c].fillna(0).astype('float32')

print(f'  AOI busyness range: [{model_df["aoi_busyness"].min():.2f}, {model_df["aoi_busyness"].max():.2f}]')
print(f'  AOI avg duration range: [{model_df["aoi_avg_duration"].min():.1f}, {model_df["aoi_avg_duration"].max():.1f}] min')

del train_data
gc.collect()

## 9. Calendar & Cyclical Temporal Features

Deterministic functions of the timestamp — no leakage risk.

**Why cyclical encoding:** Trees treat `hour=23` and `hour=0` as maximally distant.
Sin/cos encoding places them correctly as neighbors on a circle.

In [ ]:
print('Phase 6: Calendar & cyclical features...')

model_df['hour']         = model_df['bucket_hour'].dt.hour.astype('int8')
model_df['dow']          = model_df['bucket_hour'].dt.dayofweek.astype('int8')
model_df['month']        = model_df['bucket_hour'].dt.month.astype('int8')
model_df['is_weekend']   = model_df['dow'].isin([5, 6]).astype('int8')
model_df['day_of_month'] = model_df['bucket_hour'].dt.day.astype('int8')
model_df['week_of_year'] = model_df['bucket_hour'].dt.isocalendar().week.values.astype('int8')

# Cyclical encoding
model_df['hour_sin']  = np.sin(2 * np.pi * model_df['hour'] / 24).astype('float32')
model_df['hour_cos']  = np.cos(2 * np.pi * model_df['hour'] / 24).astype('float32')
model_df['dow_sin']   = np.sin(2 * np.pi * model_df['dow'] / 7).astype('float32')
model_df['dow_cos']   = np.cos(2 * np.pi * model_df['dow'] / 7).astype('float32')
model_df['month_sin'] = np.sin(2 * np.pi * (model_df['month'] - 1) / 12).astype('float32')
model_df['month_cos'] = np.cos(2 * np.pi * (model_df['month'] - 1) / 12).astype('float32')

# Interaction features
model_df['hour_sin_x_weekend'] = (model_df['hour_sin'] * model_df['is_weekend']).astype('float32')
model_df['hour_cos_x_weekend'] = (model_df['hour_cos'] * model_df['is_weekend']).astype('float32')
model_df['is_peak_x_weekend']  = (model_df['is_peak_hour'] * model_df['is_weekend']).astype('int8')

# Holiday flags (Chinese holidays in Jun-Sep 2022)
dates = model_df['bucket_hour'].dt.normalize()
dragon_boat = pd.to_datetime(['2022-06-03', '2022-06-04', '2022-06-05'])
mid_autumn  = pd.to_datetime(['2022-09-10', '2022-09-11', '2022-09-12'])
model_df['is_holiday'] = dates.isin(dragon_boat | mid_autumn).astype('int8')

print(f'  Calendar features added. Holiday rows: {model_df["is_holiday"].sum():,}')

## 10. AOI Spatial Features & Nearest-Neighbor Setup

**Improvement over old pipeline:** Instead of treating lat/lng as raw independent features,
we compute:
1. AOI centroids from delivery coordinates
2. AOI geographic spread (zone size proxy)
3. Pairwise distances between AOI centroids
4. K-nearest-neighbor lists for distance-weighted spatial features

In [ ]:
# AOI centroids from raw delivery coordinates
aoi_centroids = (
    df.groupby('aoi_id')[['lng', 'lat']]
    .mean()
    .rename(columns={'lng': 'aoi_lng', 'lat': 'aoi_lat'})
    .reset_index()
)

# AOI geographic spread
aoi_spread = (
    df.groupby('aoi_id')[['lng', 'lat']]
    .std()
    .rename(columns={'lng': 'aoi_lng_spread', 'lat': 'aoi_lat_spread'})
    .reset_index()
)
aoi_spread['aoi_area_proxy'] = np.sqrt(
    aoi_spread['aoi_lng_spread'].fillna(0)**2 + aoi_spread['aoi_lat_spread'].fillna(0)**2
).astype('float32')

model_df = model_df.merge(aoi_centroids, on='aoi_id', how='left')
model_df = model_df.merge(aoi_spread[['aoi_id', 'aoi_area_proxy']], on='aoi_id', how='left')
model_df['aoi_lng'] = model_df['aoi_lng'].astype('float32')
model_df['aoi_lat'] = model_df['aoi_lat'].astype('float32')
model_df['aoi_area_proxy'] = model_df['aoi_area_proxy'].fillna(0).astype('float32')

# Pairwise distances & nearest neighbors
print('Computing pairwise AOI distances...')
aoi_ids = aoi_centroids['aoi_id'].values
aoi_lngs = aoi_centroids['aoi_lng'].values
aoi_lats = aoi_centroids['aoi_lat'].values
n_aois = len(aoi_ids)

lng_i, lng_j = np.meshgrid(aoi_lngs, aoi_lngs, indexing='ij')
lat_i, lat_j = np.meshgrid(aoi_lats, aoi_lats, indexing='ij')
dist_matrix = haversine_km(lng_i.ravel(), lat_i.ravel(), lng_j.ravel(), lat_j.ravel()).reshape(n_aois, n_aois)

K = CONFIG['n_neighbors']
MAX_KM = CONFIG['neighbor_max_km']

aoi_neighbors = {}
for i, aoi in enumerate(aoi_ids):
    dists = dist_matrix[i]
    mask = (dists > 0) & (dists <= MAX_KM)
    valid_idx = np.where(mask)[0]
    if len(valid_idx) > 0:
        sorted_idx = valid_idx[np.argsort(dists[valid_idx])][:K]
        aoi_neighbors[aoi] = [(aoi_ids[j], dists[j]) for j in sorted_idx]
    else:
        aoi_neighbors[aoi] = []

avg_neighbors = np.mean([len(v) for v in aoi_neighbors.values()])
print(f'  AOIs: {n_aois}, Avg neighbors: {avg_neighbors:.1f}')
print(f'  AOIs with 0 neighbors: {sum(1 for v in aoi_neighbors.values() if len(v) == 0)}')

del df_raw, dist_matrix, lng_i, lng_j, lat_i, lat_j
gc.collect()

## 11. Per-AOI Lag, Rolling, Momentum & Operational Time-Series Features

**Key improvement:** The old pipeline applied lags/rolling ONLY to `demand_count`. Now we also
apply them to key operational statistics:
- `delivery_duration_min_mean` — lagged average delivery time
- `n_unique_couriers` — lagged courier availability
- `orders_per_courier` — lagged workload intensity

**Why lag operational stats?** "Yesterday at this hour, this AOI had 10 orders with avg 90-min
deliveries and 2 couriers" is a very different signal than just "10 orders". The operational
lags let the model learn from the *full state* of the delivery system.

**Zero-demand features (safe version):**
- `zero_ratio_24` — fraction of previous 24 hours with zero demand (uses shift, no leakage)
- `consec_zeros` — consecutive zero-demand hours before this one
- `is_nonzero` is **NOT included** — direct function of target value

In [ ]:
print('Phase 7: Per-AOI lag/rolling/momentum features...')

LAGS = CONFIG['lags']
ROLL_WINDOWS = CONFIG['roll_windows']
OPERATIONAL_LAG_COLS = ['delivery_duration_min_mean', 'n_unique_couriers', 'orders_per_courier']

def add_temporal_features(g):
    '''Compute temporal features for one AOI hourly series (sorted by bucket_hour).'''
    g = g.copy()
    d = g['demand_count']
    s = d.shift(1)  # shifted by 1 to prevent current-hour leakage

    # Demand lag features
    for lag in LAGS:
        g[f'lag_{lag}'] = d.shift(lag).astype('float32')

    # Operational lag features (lag 1 and lag 24)
    for col in OPERATIONAL_LAG_COLS:
        if col in g.columns:
            g[f'{col}_lag1']  = g[col].shift(1).astype('float32')
            g[f'{col}_lag24'] = g[col].shift(24).astype('float32')

    # Rolling demand statistics at multiple timescales
    for w in ROLL_WINDOWS:
        r = s.rolling(w, min_periods=w)
        g[f'roll_{w}_mean']  = r.mean().astype('float32')
        g[f'roll_{w}_std']   = r.std().astype('float32')
        g[f'roll_{w}_min']   = r.min().astype('float32')
        g[f'roll_{w}_max']   = r.max().astype('float32')
        g[f'roll_{w}_range'] = (g[f'roll_{w}_max'] - g[f'roll_{w}_min']).astype('float32')

    # Rolling operational stats (24h window)
    if 'delivery_duration_min_mean' in g.columns:
        dur_shifted = g['delivery_duration_min_mean'].shift(1)
        g['roll_24_duration_mean'] = dur_shifted.rolling(24, min_periods=24).mean().astype('float32')

    if 'n_unique_couriers' in g.columns:
        cour_shifted = g['n_unique_couriers'].shift(1)
        g['roll_24_couriers_mean'] = cour_shifted.rolling(24, min_periods=24).mean().astype('float32')

    # Demand momentum: recent vs same-hour-yesterday
    g['momentum_1_24'] = (g['lag_1'] - g['lag_24']).astype('float32')

    # Demand acceleration: change-of-change
    g['acceleration'] = (
        (g['lag_1'] - g['lag_24']) - (g['lag_24'] - g['lag_48'])
    ).astype('float32')

    # Ratio features
    roll_24 = g['roll_24_mean'].replace(0, np.nan)
    roll_168 = g['roll_168_mean'].replace(0, np.nan)
    g['lag1_over_roll24']  = (g['lag_1'] / roll_24).fillna(1.0).astype('float32')
    g['lag1_over_roll168'] = (g['lag_1'] / roll_168).fillna(1.0).astype('float32')

    # Trend: short-term vs long-term baseline
    g['trend_24_168'] = (g['roll_24_mean'] / roll_168).fillna(1.0).astype('float32')
    g['trend_6_24']   = (g['roll_6_mean'] / roll_24).fillna(1.0).astype('float32')

    # Zero-demand features (NO is_nonzero)
    zeros_in_window = (s == 0).astype('float32')
    g['zero_ratio_24'] = zeros_in_window.rolling(24, min_periods=24).mean().astype('float32')

    # Consecutive zero hours preceding this one
    is_zero = (s == 0).astype('float32')
    cumsum_zeros = is_zero.cumsum()
    reset_at_nonzero = cumsum_zeros.where(~is_zero.astype(bool)).ffill().fillna(0)
    g['consec_zeros'] = (cumsum_zeros - reset_at_nonzero).astype('int16')

    return g

model_df = (
    model_df
    .groupby(['city', 'region_id', 'aoi_id'], group_keys=False)
    .apply(add_temporal_features, include_groups=False)
)
print(f'  Temporal features done. Shape: {model_df.shape}')

## 12. Distance-Weighted Spatial Neighbor Features

**Old approach:** Average demand across all AOIs in the same `region_id` — treats every AOI
in the region equally and misses cross-region neighbors.

**New approach:** For each AOI, find K nearest geographic neighbors and compute
distance-weighted averages of their lagged demand and operational stats.

**Why distance-weighted is better:**
- An AOI 500m away is much more relevant than one 8km away in the same region
- Cross-region neighbors are captured
- Inverse-distance weighting naturally prioritizes the closest neighbors

**Features:** `neighbor_demand_lag1`, `neighbor_demand_lag24`, `neighbor_duration_lag1`,
`aoi_vs_neighbors` (hotspot detection)

In [ ]:
print('Phase 8: Distance-weighted neighbor features...')

# Build pivot tables for fast neighbor lookups
lag1_pivot = model_df.pivot_table(
    index='bucket_hour', columns='aoi_id', values='lag_1', aggfunc='first'
)
lag24_pivot = model_df.pivot_table(
    index='bucket_hour', columns='aoi_id', values='lag_24', aggfunc='first'
)

dur_lag1_col = 'delivery_duration_min_mean_lag1'
dur_pivot = None
if dur_lag1_col in model_df.columns:
    dur_pivot = model_df.pivot_table(
        index='bucket_hour', columns='aoi_id', values=dur_lag1_col, aggfunc='first'
    )

# Compute distance-weighted averages for each AOI
neighbor_demand_lag1 = {}
neighbor_demand_lag24 = {}
neighbor_duration_lag1 = {}

for aoi, neighbors in aoi_neighbors.items():
    if not neighbors or aoi not in lag1_pivot.columns:
        continue

    neighbor_ids = [n[0] for n in neighbors if n[0] in lag1_pivot.columns]
    distances    = [n[1] for n in neighbors if n[0] in lag1_pivot.columns]

    if not neighbor_ids:
        continue

    # Inverse-distance weights
    inv_dist = np.array([1.0 / max(d, 0.01) for d in distances])
    weights = inv_dist / inv_dist.sum()

    neighbor_lag1_clean  = np.nan_to_num(lag1_pivot[neighbor_ids].values, 0)
    neighbor_lag24_clean = np.nan_to_num(lag24_pivot[neighbor_ids].values, 0)

    neighbor_demand_lag1[aoi]  = pd.Series(neighbor_lag1_clean @ weights, index=lag1_pivot.index)
    neighbor_demand_lag24[aoi] = pd.Series(neighbor_lag24_clean @ weights, index=lag24_pivot.index)

    if dur_pivot is not None and all(n in dur_pivot.columns for n in neighbor_ids):
        dur_vals = np.nan_to_num(dur_pivot[neighbor_ids].values, 0)
        neighbor_duration_lag1[aoi] = pd.Series(dur_vals @ weights, index=dur_pivot.index)

# Convert to DataFrames and merge
if neighbor_demand_lag1:
    nbr_lag1_df = pd.DataFrame(neighbor_demand_lag1).stack().reset_index()
    nbr_lag1_df.columns = ['bucket_hour', 'aoi_id', 'neighbor_demand_lag1']

    nbr_lag24_df = pd.DataFrame(neighbor_demand_lag24).stack().reset_index()
    nbr_lag24_df.columns = ['bucket_hour', 'aoi_id', 'neighbor_demand_lag24']

    model_df = model_df.merge(nbr_lag1_df, on=['aoi_id', 'bucket_hour'], how='left')
    model_df = model_df.merge(nbr_lag24_df, on=['aoi_id', 'bucket_hour'], how='left')

    if neighbor_duration_lag1:
        nbr_dur_df = pd.DataFrame(neighbor_duration_lag1).stack().reset_index()
        nbr_dur_df.columns = ['bucket_hour', 'aoi_id', 'neighbor_duration_lag1']
        model_df = model_df.merge(nbr_dur_df, on=['aoi_id', 'bucket_hour'], how='left')

# Fill missing neighbor features
for c in ['neighbor_demand_lag1', 'neighbor_demand_lag24', 'neighbor_duration_lag1']:
    if c in model_df.columns:
        model_df[c] = model_df[c].fillna(0).astype('float32')

# AOI demand relative to neighbors (hotspot detection)
if 'neighbor_demand_lag1' in model_df.columns:
    model_df['aoi_vs_neighbors'] = (
        model_df['lag_1'] / model_df['neighbor_demand_lag1'].replace(0, np.nan)
    ).fillna(1.0).clip(-10, 10).astype('float32')
else:
    model_df['aoi_vs_neighbors'] = np.float32(1.0)

# Also keep region-level averages as backup
region_lag1 = (
    model_df.groupby(['region_id', 'bucket_hour'])['lag_1']
    .mean().rename('region_lag1_mean').reset_index()
)
region_lag24 = (
    model_df.groupby(['region_id', 'bucket_hour'])['lag_24']
    .mean().rename('region_lag24_mean').reset_index()
)
model_df = model_df.merge(region_lag1, on=['region_id', 'bucket_hour'], how='left')
model_df = model_df.merge(region_lag24, on=['region_id', 'bucket_hour'], how='left')
model_df['region_lag1_mean']  = model_df['region_lag1_mean'].fillna(0).astype('float32')
model_df['region_lag24_mean'] = model_df['region_lag24_mean'].fillna(0).astype('float32')

del lag1_pivot, lag24_pivot
if dur_pivot is not None:
    del dur_pivot
gc.collect()

print(f'  Neighbor features done. Shape: {model_df.shape}')

## 13. Drop Warm-Up Rows, Final Column Ordering & Save

The widest rolling window (168h = 1 week) needs 168 prior hours of data. Rows before that
have NaN features and must be dropped.

In [ ]:
print('Phase 9: Final cleanup...')

# Drop warm-up rows
warmup_col = f'roll_{max(ROLL_WINDOWS)}_mean'
before = len(model_df)
model_df = model_df.dropna(subset=[warmup_col]).reset_index(drop=True)
print(f'  Dropped {before - len(model_df):,} warm-up rows -> {len(model_df):,} remaining')

# Fill remaining NaN
fill_cols = [c for c in model_df.columns
             if model_df[c].isna().any() and model_df[c].dtype in ['float32', 'float64']]
for c in fill_cols:
    model_df[c] = model_df[c].fillna(0).astype('float32')

# Column ordering by feature group
id_cols      = ['city', 'region_id', 'aoi_id', 'aoi_type', 'bucket_hour']
meta_cols    = ['split']
target_col   = ['demand_count']

operational_cols = [
    'delivery_duration_min_mean', 'delivery_duration_min_std',
    'delivery_duration_min_min', 'delivery_duration_min_max', 'duration_range',
    'trip_distance_km_mean', 'trip_distance_km_std',
    'trip_distance_km_min', 'trip_distance_km_max', 'distance_range',
    'order_displacement_km_mean', 'order_displacement_km_std',
    'n_unique_couriers', 'orders_per_courier', 'spatial_dispersion',
]

calendar_cols = ['hour', 'dow', 'month', 'is_weekend', 'day_of_month', 'week_of_year', 'is_holiday']
cyclical_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
peak_cols     = ['is_peak_hour']
interaction_cols = ['hour_sin_x_weekend', 'hour_cos_x_weekend', 'is_peak_x_weekend']

spatial_static_cols = ['aoi_lng', 'aoi_lat', 'aoi_area_proxy', 'aoi_busyness',
                       'aoi_avg_duration', 'aoi_avg_distance', 'aoi_avg_couriers']

lag_cols = [f'lag_{l}' for l in LAGS]

op_lag_cols = []
for col in OPERATIONAL_LAG_COLS:
    for suffix in ['_lag1', '_lag24']:
        candidate = f'{col}{suffix}'
        if candidate in model_df.columns:
            op_lag_cols.append(candidate)

rolling_cols = []
for w in ROLL_WINDOWS:
    rolling_cols += [f'roll_{w}_{s}' for s in ['mean', 'std', 'min', 'max', 'range']]

rolling_op_cols = [c for c in ['roll_24_duration_mean', 'roll_24_couriers_mean']
                   if c in model_df.columns]

momentum_cols = ['momentum_1_24', 'acceleration', 'lag1_over_roll24',
                 'lag1_over_roll168', 'trend_24_168', 'trend_6_24']

zero_cols = ['zero_ratio_24', 'consec_zeros']

neighbor_cols = [c for c in [
    'neighbor_demand_lag1', 'neighbor_demand_lag24', 'neighbor_duration_lag1',
    'aoi_vs_neighbors', 'region_lag1_mean', 'region_lag24_mean'
] if c in model_df.columns]

all_feature_cols = (operational_cols + calendar_cols + cyclical_cols + peak_cols
                    + interaction_cols + spatial_static_cols + lag_cols + op_lag_cols
                    + rolling_cols + rolling_op_cols + momentum_cols + zero_cols + neighbor_cols)

# Verify all columns exist
missing = [c for c in all_feature_cols if c not in model_df.columns]
if missing:
    print(f'  WARNING: Missing columns (skipped): {missing}')
    all_feature_cols = [c for c in all_feature_cols if c in model_df.columns]

all_cols = id_cols + meta_cols + target_col + all_feature_cols
model_df = model_df[all_cols]

# Leakage validation
assert 'is_nonzero' not in model_df.columns, 'LEAKAGE: is_nonzero found!'
assert 'demand_count' not in all_feature_cols, 'LEAKAGE: target in features!'

# Save
model_df.to_csv(CONFIG['output_path'], index=False)

print(f'\n{"="*70}')
print(f'Saved: {CONFIG["output_path"]}')
print(f'Final shape: {model_df.shape}')
print(f'Total features: {len(all_feature_cols)}')
print(f'\nFeature breakdown:')
print(f'  Operational aggregation: {len(operational_cols):3d}  -- duration/distance/courier stats per AOI-hour')
print(f'  Calendar & cyclical:     {len(calendar_cols) + len(cyclical_cols):3d}  -- deterministic from timestamp')
print(f'  Peak & interaction:      {len(peak_cols) + len(interaction_cols):3d}  -- train-derived peak hours x weekend')
print(f'  Spatial (static):        {len(spatial_static_cols):3d}  -- AOI location, size, baseline profile')
print(f'  Demand lags:             {len(lag_cols):3d}  -- autoregressive features')
print(f'  Operational lags:        {len(op_lag_cols):3d}  -- lagged duration/courier stats')
print(f'  Rolling demand:          {len(rolling_cols):3d}  -- multi-timescale demand summaries')
print(f'  Rolling operational:     {len(rolling_op_cols):3d}  -- trending duration/courier availability')
print(f'  Momentum & trend:        {len(momentum_cols):3d}  -- demand acceleration/deceleration')
print(f'  Zero-demand:             {len(zero_cols):3d}  -- dormancy patterns (no is_nonzero)')
print(f'  Spatial neighbors:       {len(neighbor_cols):3d}  -- distance-weighted nearby AOI stats')

## 14. Validation & Feature Summary

In [ ]:
# Verify output
preview = pd.read_csv(CONFIG['output_path'], nrows=3)
print(f'Output columns ({len(preview.columns)}):')
for i, c in enumerate(preview.columns):
    print(f'  {i:3d}. {c}')

# Split distribution after warm-up drop
print(f'\nSplit distribution:')
print(model_df['split'].value_counts().sort_index())

# NaN check
nan_counts = model_df[all_feature_cols].isna().sum()
if nan_counts.sum() > 0:
    print(f'\nWARNING: {nan_counts.sum()} remaining NaN values:')
    print(nan_counts[nan_counts > 0])
else:
    print(f'\nNo NaN values in features.')

# Feature correlation with target (train only, sanity check)
train_slice = model_df[model_df['split'] == 'train']
correlations = train_slice[all_feature_cols + ['demand_count']].corr()['demand_count'].drop('demand_count')
top_corr = correlations.abs().sort_values(ascending=False).head(15)
print(f'\nTop 15 features by |correlation| with demand_count (train only):')
for feat, corr in top_corr.items():
    actual_corr = correlations[feat]
    print(f'  {feat:35s}: {actual_corr:+.4f}')

# Memory
mem_gb = model_df.memory_usage(deep=True).sum() / 1e9
print(f'\nFinal memory usage: {mem_gb:.2f} GB')

# Final summary
print(f'\n{"="*70}')
print('FEATURE ENGINEERING PIPELINE COMPLETE')
print(f'{"="*70}')
print(f'  AOIs:          {model_df["aoi_id"].nunique()}')
print(f'  Total rows:    {len(model_df):,}')
print(f'  Features:      {len(all_feature_cols)}')
print(f'  Date range:    {model_df["bucket_hour"].min()} -> {model_df["bucket_hour"].max()}')
print(f'  Output:        {CONFIG["output_path"]}')
print()
print('LEAKAGE SAFEGUARDS:')
print('  [REMOVED]     is_nonzero -- direct target leakage')
print('  [TRAIN-ONLY]  is_peak_hour -- peak hours derived from train split only')
print('  [TRAIN-ONLY]  aoi_busyness -- AOI baseline activity from train split only')
print('  [TRAIN-ONLY]  aoi_avg_duration/distance/couriers -- AOI profile from train only')
print('  [SAFE]        All lag/rolling features use shift() -- no current-hour leakage')
print('  [SAFE]        Calendar/cyclical features are deterministic from timestamp')
print()
print('NEW SIGNAL CATEGORIES (vs old pipeline):')
print('  [NEW] Order-level operational stats -- duration, distance, courier metrics per AOI-hour')
print('  [NEW] Operational lags -- lagged duration & courier stats (not just demand count)')
print('  [NEW] Distance-weighted neighbor features -- nearby AOI demand/operations')
print('  [NEW] AOI operational profile -- baseline duration/distance/couriers per AOI')
print('  [NEW] AOI geographic spread -- zone size proxy from coordinate dispersion')
print('  [NEW] Holiday flags -- Chinese public holidays in data range')